# LLM Experiment-6

In [1]:
!pip install groq ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 58.9 MB/s eta 0:00:00


In [2]:
import os
from groq import Groq
from google.colab import userdata
from getpass import getpass

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

In [3]:
from ddgs import DDGS

def web_search(query):
    """Fetches real-time information from the internet."""
    print(f"  [Searching the web for: {query}...]")
    with DDGS() as ddgs:
        results = [r['body'] for r in ddgs.text(query, max_results=3)]
        return "\n".join(results)

print("Web search tool defined.")

Web search tool defined.


In [4]:
SYSTEM_PROMPT = """
You are a helpful AI assistant with access to a web search tool.
When a user asks a question, follow these steps:
1. If you know the answer and it doesn't require real-time info, provide the answer.
2. If you need current information or confirmation, output exactly: SEARCH: <your search query>
3. After receiving search results, provide a concise FINAL ANSWER based on the results.

Example:
User: Who won the game last night?
Assistant: SEARCH: latest sports scores for [Team]
"""

def simple_agent(user_query):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            temperature=0.2
        )
    except Exception as e:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            temperature=0.2
        )

    llm_text = response.choices[0].message.content

    if "SEARCH:" in llm_text:
        search_query = llm_text.split("SEARCH:")[1].strip()
        search_results = web_search(search_query)

        messages.append({"role": "assistant", "content": llm_text})
        messages.append({"role": "user", "content": f"Search Results: {search_results}"})

        final_response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            temperature=0.2
        )
        return final_response.choices[0].message.content

    return llm_text

print("Agent workflow function initialized with updated models.")

Agent workflow function initialized with updated models.


In [5]:

user_input = "Who is the current CEO of Tesla and what was their most recent major announcement?"

print(f"Question: {user_input}")
answer = simple_agent(user_input)
print("\n--- FINAL ANSWER ---")
print(answer)

Question: Who is the current CEO of Tesla and what was their most recent major announcement?
  [Searching the web for: current CEO of Tesla and latest major announcement...]

--- FINAL ANSWER ---
FINAL ANSWER: The current CEO of Tesla is Elon Musk. Their most recent major announcement was the decision to pull the Model S and Model X from its lineup, as announced during the last quarterly earnings call.


In [6]:
test_queries = [
    "What is photosynthesis?",
    "Who is the CEO of Tesla?",
    "Latest news about AI advancements?",
    "Explain the theory of relativity."
]

for q in test_queries:
    print(f"\nQUERY: {q}")
    answer = simple_agent(q)
    print(f"ANSWER: {answer}")
    print("-" * 60)


QUERY: What is photosynthesis?
ANSWER: Photosynthesis is the process by which plants, algae, and some bacteria convert light energy from the sun into chemical energy in the form of organic compounds, such as glucose. This process occurs in specialized organelles called chloroplasts, which contain the pigment chlorophyll. Chlorophyll absorbs light energy, which is then used to convert carbon dioxide and water into glucose and oxygen.

The overall equation for photosynthesis is:

6 CO2 (carbon dioxide) + 6 H2O (water) + light energy → C6H12O6 (glucose) + 6 O2 (oxygen)

Photosynthesis is essential for life on Earth, as it provides the energy and organic compounds needed to support the food chain. It also helps regulate the Earth's atmosphere by removing carbon dioxide and releasing oxygen.
------------------------------------------------------------

QUERY: Who is the CEO of Tesla?
ANSWER: The CEO of Tesla is Elon Musk.
------------------------------------------------------------

QUERY:

In [8]:
def run_interactive_menu():
    print("==========================================")
    print("   WELCOME TO THE WEB-ENABLED QA AGENT   ")
    print("==========================================")
    print("Type 'exit' to stop the session.\n")

    while True:
        user_input = input("Enter your Question: ")

        if user_input.lower() in ['exit', 'quit', 'stop']:
            print("\nShutting down. Goodbye!")
            break

        if not user_input.strip():
            continue

        print("\nThinking...")
        answer = simple_agent(user_input)

        print("\n--- RESPONSE ---")
        print(answer)
        print("----------------\n")

run_interactive_menu()

   WELCOME TO THE WEB-ENABLED QA AGENT   
Type 'exit' to stop the session.

Enter your Question: Tell me something about Dhurandhar.

Thinking...

--- RESPONSE ---
Mahadev Dhurandhar (1867-1944) was a renowned Indian painter and illustrator from Maharashtra. He is best known for his work in the fields of portrait painting, landscape painting, and illustration, particularly in the context of Indian culture and society during the late 19th and early 20th centuries.

Dhurandhar's artwork often featured scenes from everyday life, mythology, and historical events. He was one of the first Indian artists to gain recognition for his work in Western-style painting, and his contributions played a significant role in promoting Indian art and culture.

Some of his notable works include illustrations for various Indian publications, such as the 'Indian Punch' and 'The Illustrated Weekly of India', as well as portraits of prominent Indian personalities. Dhurandhar's style blended traditional Indian 